In [1]:
from datasets import load_dataset
from transformers import BertTokenizer

### 加载编码器工具

In [3]:
import os


os.environ['http_proxy'] = '127.0.0.1:10809'
os.environ['https_proxy'] = '127.0.0.1:10809'

In [5]:
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')
tokenizer

BertTokenizer(name_or_path='bert-base-chinese', vocab_size=21128, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'})

In [6]:
# 试编码句子, 观察输出
out = tokenizer.batch_encode_plus(
    batch_text_or_text_pairs=['从明天起，做一个幸福的人。', '喂马， 劈柴，周游世界。'],
    truncation=True,
    padding='max_length',
    max_length=17,
    return_tensors='pt',
    return_length=True
)

In [7]:
for k, v in out.items():
    print(k, v.shape)

input_ids torch.Size([2, 17])
token_type_ids torch.Size([2, 17])
length torch.Size([2])
attention_mask torch.Size([2, 17])


In [8]:
for k, v in out.items():
    print(k, v)

input_ids tensor([[ 101,  794, 3209, 1921, 6629, 8024,  976,  671,  702, 2401, 4886, 4638,
          782,  511,  102,    0,    0],
        [ 101, 1585, 7716, 8024, 1207, 3395, 8024, 1453, 3952,  686, 4518,  511,
          102,    0,    0,    0,    0]])
token_type_ids tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
length tensor([15, 13])
attention_mask tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]])


In [9]:
# 把编码还原成句子
print(tokenizer.decode(out['input_ids'][0]))

[CLS] 从 明 天 起 ， 做 一 个 幸 福 的 人 。 [SEP] [PAD] [PAD]


### 定义数据集

In [10]:
import torch

In [11]:
from datasets import load_from_disk

In [14]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, split):
        self.dataset = load_from_disk('./data/ChnSentiCorp')[split]
        
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, i):
        text = self.dataset[i]['text']
        label = self.dataset[i]['label']
        
        return text, label
    
dataset = Dataset('train')

In [15]:
len(dataset)

9600

In [16]:
dataset[20]

('非常不错，服务很好，位于市中心区，交通方便，不过价格也高！', 1)

### 定义计算设备

In [18]:
device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
    
device

'cuda'

In [19]:
# 更简洁的写法, 更python的写法， pythonic
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [20]:
device

'cuda'

### 数据整理函数

In [21]:
def collate_fn(data):
    sents = [i[0] for i in data]
    labels = [i[1] for i in data]
    
    # 编码
    data = tokenizer.batch_encode_plus(batch_text_or_text_pairs=sents,
                               truncation=True,
                               padding='max_length',
                               max_length=500,
                               return_tensors='pt',
                               return_length=True)
    
    # input_ids: 编码之后的数字
    # attention_mask: 0的位置是不需要计算attention的， 1的位置表示要计算attention
    # token_type_ids: token的类型， 0表示第一个句子， 1表示第二个句子。 
    input_ids = data['input_ids']
    attention_mask = data['attention_mask']
    token_type_ids = data['token_type_ids']
    labels = torch.LongTensor(labels)
    
    # 把数据拷贝到计算设备上
    # 这个操作也可以在训练的时候做。 
    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    token_type_ids = token_type_ids.to(device)
    labels = labels.to(device)
    
    return input_ids, attention_mask, token_type_ids, labels


In [22]:
# 测试一下整理函数
# 先模拟一批数据

data = [
    ('你站在桥上看风景', 1), 
    ('看风景的人在楼上看你', 0),
    ('明月装饰了你的窗', 1), 
    ('你装饰了别人的梦', 0)
]

In [25]:
input_ids, attention_mask, token_type_ids, labels = collate_fn(data)
input_ids.shape, attention_mask.shape, token_type_ids.shape

(torch.Size([4, 500]), torch.Size([4, 500]), torch.Size([4, 500]))

In [26]:
# 数据加载器
loader = torch.utils.data.DataLoader(dataset=dataset,
                           batch_size=16,
                           collate_fn=collate_fn,
                           shuffle=True,
                           drop_last=True)

len(loader)

600

In [27]:
# 查看数据样例
for i, (input_ids, attention_mask, token_type_ids, labels) in enumerate(loader):
    break
    
input_ids.shape, attention_mask.shape, token_type_ids.shape, labels

(torch.Size([16, 500]),
 torch.Size([16, 500]),
 torch.Size([16, 500]),
 tensor([0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0], device='cuda:0'))

In [28]:
# 加载预训练模型
from transformers import BertModel

pretrained = BertModel.from_pretrained('bert-base-chinese')

D:\.venv\lib\site-packages\huggingface_hub\file_download.py:133: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SupercoldZzz\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of the model checkpoint at bert-base-chinese were not used when initializing BertModel: ['cls.seq_relationship.bias', 'cls.predictions.decoder.we

In [29]:
# 计算参数量
sum(i.numel() for i in pretrained.parameters())

102267648

In [30]:
# 冻结参数
for param in pretrained.parameters():
    param.requires_grad_(False)

In [31]:
# 预训练模型试算
pretrained.to(device)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(21128, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0): BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          

In [32]:
out = pretrained(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)

In [34]:
out.last_hidden_state.shape

torch.Size([16, 500, 768])

In [35]:
# 定义下游任务模型
class Model(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = torch.nn.Linear(in_features=768, out_features=2)
        
    def forward(self, input_ids, attention_mask, token_type_ids):
        # 使用预训练模型抽取数据特征
        with torch.no_grad():
            out = pretrained(input_ids=input_ids,
                            attention_mask=attention_mask,
                            token_type_ids=token_type_ids)
            
        # 对抽取的特征只取第一个字的结果做分类。 bert中第一个字是《cls》
        out = self.fc(out.last_hidden_state[:, 0])
        out = out.softmax(dim=1)
        return out
    
model = Model()

# 拷贝到GPU上
model.to(device)

Model(
  (fc): Linear(in_features=768, out_features=2, bias=True)
)

In [38]:
# 试算
model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)

tensor([[0.3795, 0.6205],
        [0.3369, 0.6631],
        [0.2953, 0.7047],
        [0.3959, 0.6041],
        [0.3804, 0.6196],
        [0.3818, 0.6182],
        [0.5551, 0.4449],
        [0.5161, 0.4839],
        [0.3945, 0.6055],
        [0.3662, 0.6338],
        [0.4015, 0.5985],
        [0.5394, 0.4606],
        [0.3916, 0.6084],
        [0.3610, 0.6390],
        [0.3826, 0.6174],
        [0.6825, 0.3175]], device='cuda:0', grad_fn=<SoftmaxBackward0>)

In [41]:
# 训练
from transformers import AdamW
from transformers.optimization import get_scheduler


def train():
    # 定义优化器
    optimizer = AdamW(model.parameters(), lr=5e-4)
    
    # 定义损失函数
    criterion = torch.nn.CrossEntropyLoss()
    
    # 定义学习率调节器
    scheduler = get_scheduler(name='linear',
                 num_warmup_steps=0,
                 num_training_steps=len(loader),
                 optimizer=optimizer)
    
    # 切换到训练模式
    model.train()
    
    
    # 按批次遍历训练集中的数据
    for i, (input_ids, attention_mask, token_type_ids, labels) in enumerate(loader):
        # 模型计算
        out = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
    
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
        # 输出各项数据
        if i % 10 == 0:
            out = out.argmax(dim=1)
            accuracy = (out == labels).sum().item() / len(labels)
            lr = optimizer.state_dict()['param_groups'][0]['lr']
            print(i, loss.item(), lr, accuracy)

In [42]:
train()

D:\.venv\lib\site-packages\transformers\optimization.py:310: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  FutureWarning,


0 0.6679812669754028 0.0004991666666666666 0.625
10 0.6479864716529846 0.0004908333333333334 0.6875
20 0.5717204809188843 0.0004825 0.75
30 0.5883175134658813 0.0004741666666666667 0.75
40 0.5785955786705017 0.00046583333333333334 0.6875
50 0.4522709250450134 0.0004575 1.0
60 0.515662431716919 0.00044916666666666667 0.875
70 0.4768165946006775 0.0004408333333333334 0.875
80 0.5174072980880737 0.0004325 0.875
90 0.441919207572937 0.0004241666666666667 0.9375
100 0.46389296650886536 0.0004158333333333333 0.875
110 0.49081146717071533 0.0004075 0.8125
120 0.6470122933387756 0.0003991666666666667 0.625
130 0.5442882776260376 0.0003908333333333333 0.75
140 0.44341689348220825 0.00038250000000000003 0.9375
150 0.43992024660110474 0.00037416666666666664 0.9375
160 0.5134100914001465 0.00036583333333333335 0.8125
170 0.3984551727771759 0.0003575 0.9375
180 0.5433549880981445 0.0003491666666666667 0.6875
190 0.5030835866928101 0.00034083333333333334 0.75
200 0.4672262966632843 0.0003325 0.9375


In [45]:
# 测试
def test():
    # 定义测试的数据加载器
    loader_test = torch.utils.data.DataLoader(dataset=Dataset('test'),
                               batch_size=32,
                               collate_fn=collate_fn,
                               shuffle=True,
                               drop_last=True)
    
    # 下游任务模型切换到测试模式
    model.eval()
    correct = 0
    total = 0
    
    # 按批次遍历测试集中的数据
    for i, (input_ids, attention_mask, token_type_ids, labels) in enumerate(loader_test):
        # 计算5个批次， 不需要全部遍历
        if i == 5:
            break
            
        print(i)
        
        # 计算
        with torch.no_grad():
            out = model(input_ids=input_ids,
                 attention_mask=attention_mask,
                 token_type_ids=token_type_ids)
            
        # 计算准确率
        out = out.argmax(dim=1)
        correct += (out == labels).sum().item()
        total += len(labels)
        
    print(correct / total)

In [46]:
test()

0
1
2
3
4
0.8625
